# 02 — Build and Run an Agent

This notebook walks you through creating an agent, sending a message, and getting a response.
Change `AGENT_NAME` below to work with any registered agent.

## Configuration

In [ ]:
# Change this to work with a different agent
AGENT_NAME = "code-helper"

# Your Foundry project connection string
CONNECTION_STRING = "<your-connection-string>"

## Install Dependencies

In [ ]:
%pip install azure-ai-agents azure-identity -q

## Connect to Foundry

In [ ]:
from azure.ai.agents import AgentsClient
from azure.identity import DefaultAzureCredential

client = AgentsClient(
    endpoint=CONNECTION_STRING,
    credential=DefaultAzureCredential()
)
print("✓ Connected to Azure AI Foundry")

## Step 1: Create an Agent

Create a simple agent with instructions and a function tool.

In [ ]:
from azure.ai.agents.models import FunctionTool

# Define a simple tool
def greet_user(name: str) -> str:
    """Greet a user by name."""
    return f"Hello, {name}! Welcome to the Foundry Agent Platform."

tool = FunctionTool(functions=[greet_user])

# Create the agent
agent = client.create_agent(
    model="gpt-4o",
    name=f"notebook-{AGENT_NAME}",
    instructions="You are a helpful assistant. Use the greet_user tool when asked to greet someone.",
    tools=tool.definitions,
)

print(f"✓ Agent created: {agent.name} (id: {agent.id})")

## Step 2: Create a Thread and Send a Message

In [ ]:
# Create a conversation thread
thread = client.threads.create()
print(f"✓ Thread created: {thread.id}")

# Post a message
message = client.messages.create(
    thread_id=thread.id,
    role="user",
    content="Hello! Can you greet a user named Alice?",
)
print(f"✓ Message posted: {message.id}")

## Step 3: Run the Agent and Get a Response

In [ ]:
from azure.ai.agents.models import MessageRole

# Create and process the run (SDK handles polling)
run = client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id,
)

print(f"Run status: {run.status}")

# Retrieve the response
last_msg = client.messages.get_last_message_text_by_role(
    thread_id=thread.id,
    role=MessageRole.AGENT,
)

response = last_msg.text.value if last_msg else "(no response)"
print(f"\nAgent response:\n{response}")

## Step 4: Cleanup

Delete the test agent and thread.

In [ ]:
client.threads.delete(thread.id)
client.delete_agent(agent.id)
print("✓ Cleanup complete — agent and thread deleted")

## Using the Deploy Script

Instead of creating agents manually in notebooks, use the deploy script for production:

```bash
# Deploy a single agent
python scripts/deploy_agent.py --agent code-helper

# Deploy all agents
python scripts/deploy_agent.py --all
```